# Modelo CA mixto con duracion de cambio de carril

## Marco teorico
El articulo parte de la idea clasica de los **modelos de automata celular (CA)** para trafico: la via se discretiza en celdas y cada vehiculo actualiza su estado por reglas locales. Esto permite capturar fenomenos emergentes (ondas de congestion, transiciones de fase) con reglas simples y computacion eficiente.

**Tres ideas clave del marco teorico**
- **Microsimulacion discreta:** cada vehiculo tiene posicion y velocidad discretas y evoluciona en pasos de $\Delta t=1\,s$.
- **Coexistencia de tipos de vehiculo:** motos y autos difieren en longitud, velocidad maxima y tiempo de cambio de carril, lo cual modifica la ocupacion espacial de la via.
- **Cambio de carril con duracion:** a diferencia del STCA clasico (cambio instantaneo), se introduce una duracion $\tau$ para representar el proceso continuo de desplazamiento lateral. Durante $\tau$ el vehiculo ocupa simultaneamente el carril de origen y el carril objetivo (doble ocupacion).

**Implicaciones teoricas**
- La doble ocupacion reduce la capacidad efectiva, especialmente a densidades medias (cerca del umbral critico).
- La penetracion de motos puede **reducir flujo en baja densidad** (por menor $v_{max}$) pero **mejorar flujo en alta densidad** (por cambios de carril mas rapidos y menor ocupacion longitudinal).
- El modelo reproduce la forma general del diagrama fundamental y es compatible con la teoria de trafico de tres fases (libre, sincronizado, congestionado).


## Parametros del modelo (segun el articulo)
La simulacion usa una via unidireccional con celdas de $1\,m$ y $\Delta t=1\,s$. Los parametros se derivan de datos NGSIM y literatura de trafico mixto.

**Geometria y discretizacion**
- Longitud de via: $L=2000\,m$ (en el articulo), aqui se mantiene la misma longitud.
- Longitud de celda: $l_{cell}=1\,m$ (permite representar longitudes reales de vehiculos).
- Numero de carriles: el articulo usa 2 carriles; aqui usamos 3 para analizar mayor heterogeneidad lateral.

**Vehiculos (heterogeneidad)**
- Longitud auto: $l_{car}=5\,m$
- Longitud moto: $l_{motor}=3\,m$
- Velocidad maxima auto: $v_{max,car}=20\,m/s$ (72 km/h)
- Velocidad maxima moto: $v_{max,motor}=15\,m/s$ (54 km/h)

**Dinamica longitudinal**
- Aceleracion discreta: $v \leftarrow \min(v+1, v_{max})$ (equivalente a $a\approx 2.5\,m/s^2$)
- Frenado maximo: $b=4\,m/s^2$ (usado en la restriccion de seguridad tipo Gipps)
- Probabilidad de frenado aleatorio: $p_{slow}=0.16$ (representa perturbaciones y variabilidad humana)

**Cambio de carril**
- Probabilidad de intentar cambio: $p_{change}=0.8$
- Duracion del cambio:
  - Motos: $\tau_m=1\,s$
  - Autos: $\tau_c\in\{0,2,4,6\}\,s$
- El caso $\tau_c=0$ reproduce el cambio instantaneo del STCA clasico.

**Penetracion de motos**
- $r\in\{0, 0.05, 0.10, 0.15, 0.20\}$ con $\tau_m=1\,s$ y $\tau_c=4\,s$

**Interpretacion fisica de parametros clave**
- $p_{slow}$: probabilidad de una desaceleracion aleatoria por percepcion, reaccion o perturbaciones externas. Captura ruido humano y genera oscilaciones de velocidad.
- $p_{change}$: probabilidad de ejecutar el cambio de carril cuando se cumplen motivacion y seguridad. Modela diferencias de agresividad y disponibilidad de oportunidades laterales.
- $b$: capacidad de frenado maximo usada en la condicion de seguridad tipo Gipps; controla cuan estricta es la aceptacion del hueco lateral.

Estos parametros permiten evaluar el efecto de **duracion de cambio de carril** y **penetracion de motos** sobre el flujo, la velocidad media y la capacidad de la via.

## Reglas del modelo (resumen)

**1) Reglas longitudinales tipo NaSch**
Para cada vehiculo en su carril actual:
1. Aceleracion: $v \leftarrow \min(v+1, v_{max})$
2. Frenado por hueco: $v \leftarrow \min(v, \text{gap})$
3. Frenado aleatorio: con prob. $p_{slow}$, $v \leftarrow \max(v-1, 0)$
4. Actualizacion de posicion: $x \leftarrow x + v$

**2) Seguridad tipo Gipps (usada en cambio de carril)**
La velocidad segura se aproxima como:
$$v_{safe} = -b\,t_r + \sqrt{(b\,t_r)^2 + v_{lead}^2 + 2b\,d}$$
donde $d$ es la distancia libre hasta la parte trasera del lider y $t_r=1\,s$.
Se exige que el seguidor en el carril objetivo pueda frenar: $v_{follower} \le v_{safe}$.

**3) Cambio de carril con duracion $\tau$**
- **Motivacion:** el vehiculo intenta cambiar si el hueco adelante en el carril objetivo es mayor que el de su carril actual y su velocidad esta limitada por el hueco actual.
- **Seguridad:** se valida hueco adelante y atras en el carril objetivo (usando la condicion de Gipps).
- **Duracion:** al iniciar el cambio, el vehiculo entra en fase de cambio por $\tau$ pasos. Durante ese tiempo ocupa **ambos carriles** (vehiculo real + vehiculo virtual).
- **Fin del cambio:** cuando $\tau=0$, el vehiculo queda solo en el carril objetivo y libera el carril de origen.


## Simulador rapido en C++ (Colab)
El siguiente bloque escribe el codigo C++ dentro del notebook, compila en Colab y genera un CSV.
En Colab no necesitas instalar nada: `g++` ya esta disponible.


In [ ]:
# Escribe el codigo C++ dentro del notebook (Colab)
cpp_code = r"""
// ca_mixed.cpp
// Simulador CA mixto con cambio de carril y duracion
// Compilar: g++ -O2 -std=c++17 ca_mixed.cpp -o ca_mixed
// Ejecutar: ./ca_mixed results.csv

#include <algorithm>
#include <cmath>
#include <fstream>
#include <iomanip>
#include <iostream>
#include <random>
#include <string>
#include <vector>

struct Vehicle {
    int id;
    int lane;
    int pos;
    int speed;
    int vmax;
    int length;
    bool isMotor;
    int lcTimer;
    int lcFrom;
};

struct Params {
    int roadLength = 2000;
    int lanes = 3;
    int vMaxCar = 20;
    int vMaxMotor = 15;
    int lenCar = 5;
    int lenMotor = 3;
    int tauCar = 4;
    int tauMotor = 1;
    double b = 4.0;
    double pChange = 0.8;
    double pSlow = 0.16;
    double reactionTime = 1.0;
};

struct Entry {
    int pos;
    int length;
    int speed;
    int vid;
};

int safe_speed_gipps(int vFollower, int vLead, int gap, double b, double reactionTime) {
    if (gap <= 0) {
        return 0;
    }
    double term = (b * reactionTime) * (b * reactionTime) + vLead * vLead + 2.0 * b * gap;
    double vSafe = -b * reactionTime + std::sqrt(std::max(0.0, term));
    if (vSafe < 0.0) {
        return 0;
    }
    return static_cast<int>(std::floor(vSafe));
}

std::vector<std::vector<Entry>> build_lane_index(const std::vector<Vehicle>& vehicles, const Params& params) {
    std::vector<std::vector<Entry>> lanes(params.lanes);
    for (const auto& v : vehicles) {
        lanes[v.lane].push_back({v.pos, v.length, v.speed, v.id});
        if (v.lcTimer > 0 && v.lcFrom >= 0) {
            lanes[v.lcFrom].push_back({v.pos, v.length, v.speed, v.id});
        }
    }
    for (auto& lane : lanes) {
        std::sort(lane.begin(), lane.end(), [](const Entry& a, const Entry& b) {
            return a.pos < b.pos;
        });
    }
    return lanes;
}

bool find_index(const std::vector<Entry>& lane, int vid, int& index) {
    for (size_t i = 0; i < lane.size(); ++i) {
        if (lane[i].vid == vid) {
            index = static_cast<int>(i);
            return true;
        }
    }
    return false;
}

int gap_ahead(const Entry& entry, const Entry* leader, int L) {
    if (!leader) {
        return L;
    }
    int raw = leader->pos - entry.pos - leader->length;
    if (raw < 0) {
        raw += L;
    }
    return std::max(0, raw);
}

int gap_behind(const Entry& entry, const Entry* follower, int L) {
    if (!follower) {
        return L;
    }
    int raw = entry.pos - entry.length - follower->pos;
    if (raw < 0) {
        raw += L;
    }
    return std::max(0, raw);
}

int decide_lane_change(const Vehicle& v, const std::vector<std::vector<Entry>>& lanesIndex, const Params& params,
                       std::mt19937& rng) {
    if (v.lcTimer > 0) {
        return -1;
    }
    const auto& laneList = lanesIndex[v.lane];
    int idx = -1;
    if (!find_index(laneList, v.id, idx)) {
        return -1;
    }
    int n = static_cast<int>(laneList.size());
    const Entry& entry = laneList[idx];
    const Entry* leader = nullptr;
    if (n > 1) {
        leader = &laneList[(idx + 1) % n];
    }
    int gapCurr = gap_ahead(entry, leader, params.roadLength);
    bool desiredLimited = gapCurr < v.speed + 1;

    std::vector<int> candidates;
    for (int delta : {-1, 1}) {
        int target = v.lane + delta;
        if (target < 0 || target >= params.lanes) {
            continue;
        }
        const auto& targetList = lanesIndex[target];
        const Entry* leaderT = nullptr;
        const Entry* followerT = nullptr;
        if (!targetList.empty()) {
            int nT = static_cast<int>(targetList.size());
            int ins = 0;
            while (ins < nT && targetList[ins].pos < entry.pos) {
                ++ins;
            }
            int leaderIdx = ins % nT;
            int followerIdx = (ins - 1 + nT) % nT;
            leaderT = &targetList[leaderIdx];
            followerT = &targetList[followerIdx];
        }
        int gapFront = gap_ahead(entry, leaderT, params.roadLength);
        int gapBack = gap_behind(entry, followerT, params.roadLength);
        bool better = gapFront > gapCurr;
        if (!(desiredLimited && better)) {
            continue;
        }
        int vLead = leaderT ? leaderT->speed : 0;
        int safeFront = safe_speed_gipps(v.speed, vLead, gapFront, params.b, params.reactionTime);
        bool safeBack = true;
        if (followerT) {
            int vFollow = followerT->speed;
            int safeFollow = safe_speed_gipps(vFollow, v.speed, gapBack, params.b, params.reactionTime);
            safeBack = vFollow <= safeFollow;
        }
        if (safeFront >= v.speed && safeBack) {
            candidates.push_back(target);
        }
    }

    if (candidates.empty()) {
        return -1;
    }
    std::uniform_real_distribution<double> uni(0.0, 1.0);
    if (uni(rng) > params.pChange) {
        return -1;
    }
    std::uniform_int_distribution<int> pick(0, static_cast<int>(candidates.size()) - 1);
    return candidates[pick(rng)];
}

void step_simulation(std::vector<Vehicle>& vehicles, const Params& params, std::mt19937& rng) {
    auto lanesIndex = build_lane_index(vehicles, params);

    for (auto& v : vehicles) {
        int target = decide_lane_change(v, lanesIndex, params, rng);
        if (target >= 0) {
            v.lcFrom = v.lane;
            v.lane = target;
            v.lcTimer = v.isMotor ? params.tauMotor : params.tauCar;
        }
    }

    lanesIndex = build_lane_index(vehicles, params);
    std::uniform_real_distribution<double> uni(0.0, 1.0);
    for (auto& v : vehicles) {
        const auto& laneList = lanesIndex[v.lane];
        int idx = -1;
        if (!find_index(laneList, v.id, idx)) {
            continue;
        }
        int n = static_cast<int>(laneList.size());
        const Entry& entry = laneList[idx];
        const Entry* leader = nullptr;
        if (n > 1) {
            leader = &laneList[(idx + 1) % n];
        }
        int gap = gap_ahead(entry, leader, params.roadLength);
        v.speed = std::min(v.speed + 1, v.vmax);
        v.speed = std::min(v.speed, gap);
        if (uni(rng) < params.pSlow) {
            v.speed = std::max(v.speed - 1, 0);
        }
        v.pos = v.pos + v.speed;
        v.pos = v.pos % params.roadLength;
    }

    for (auto& v : vehicles) {
        if (v.lcTimer > 0) {
            v.lcTimer -= 1;
            if (v.lcTimer == 0) {
                v.lcFrom = -1;
            }
        }
    }
}

std::vector<Vehicle> init_vehicles(double densityPerKm, double motorRatio, const Params& params, std::mt19937& rng) {
    int totalCells = params.roadLength;
    int target = static_cast<int>(densityPerKm * (params.roadLength / 1000.0) * params.lanes);
    std::vector<Vehicle> vehicles;
    vehicles.reserve(target);
    std::vector<std::vector<bool>> occupied(params.lanes, std::vector<bool>(totalCells, false));

    int vid = 0;
    int attempts = 0;
    std::uniform_int_distribution<int> lanePick(0, params.lanes - 1);
    std::uniform_int_distribution<int> posPick(0, totalCells - 1);
    std::uniform_real_distribution<double> uni(0.0, 1.0);

    while (static_cast<int>(vehicles.size()) < target && attempts < target * 50) {
        attempts++;
        int lane = lanePick(rng);
        bool isMotor = uni(rng) < motorRatio;
        int length = isMotor ? params.lenMotor : params.lenCar;
        int vmax = isMotor ? params.vMaxMotor : params.vMaxCar;
        int pos = posPick(rng);

        bool ok = true;
        for (int i = 0; i < length; ++i) {
            int c = (pos - i + totalCells) % totalCells;
            if (occupied[lane][c]) {
                ok = false;
                break;
            }
        }
        if (!ok) {
            continue;
        }
        for (int i = 0; i < length; ++i) {
            int c = (pos - i + totalCells) % totalCells;
            occupied[lane][c] = true;
        }
        vehicles.push_back({vid, lane, pos, vmax, vmax, length, isMotor, 0, -1});
        vid++;
    }
    return vehicles;
}

void run_fd(const std::vector<double>& densities, double motorRatio, Params params, int steps, int sample,
            std::mt19937& rng, std::vector<double>& outFlow, std::vector<double>& outSpeed) {
    outFlow.clear();
    outSpeed.clear();
    for (double d : densities) {
        auto vehicles = init_vehicles(d, motorRatio, params, rng);
        double speedAccum = 0.0;
        int sampleCount = 0;
        for (int t = 0; t < steps; ++t) {
            step_simulation(vehicles, params, rng);
            if (t >= steps - sample) {
                double meanSpeed = 0.0;
                if (!vehicles.empty()) {
                    for (const auto& v : vehicles) {
                        meanSpeed += v.speed;
                    }
                    meanSpeed /= vehicles.size();
                }
                speedAccum += meanSpeed;
                sampleCount++;
            }
        }
        double meanSpeed = sampleCount > 0 ? speedAccum / sampleCount : 0.0;
        double density = vehicles.empty() ? 0.0 : (vehicles.size() / (params.roadLength / 1000.0) / params.lanes);
        double flow = density * meanSpeed * 3.6;
        outFlow.push_back(flow);
        outSpeed.push_back(meanSpeed * 3.6);
    }
}

int main(int argc, char** argv) {
    std::string outputPath = "results.csv";
    if (argc > 1) {
        outputPath = argv[1];
    }

    Params params;
    std::mt19937 rng(7);
    std::vector<double> densities;
    for (int i = 0; i < 12; ++i) {
        double v = 5.0 + i * (115.0 / 11.0);
        densities.push_back(v);
    }

    std::ofstream out(outputPath);
    if (!out) {
        std::cerr << "No se pudo abrir el archivo de salida." << std::endl;
        return 1;
    }
    out << "experiment,tau_car,motor_ratio,density,flow,speed\n";

    std::vector<double> flows;
    std::vector<double> speeds;

    std::vector<int> tauCases = {0, 2, 4, 6};
    for (int tau : tauCases) {
        params.tauCar = tau;
        params.tauMotor = 1;
        run_fd(densities, 0.10, params, 2000, 200, rng, flows, speeds);
        for (size_t i = 0; i < densities.size(); ++i) {
            out << "tau," << tau << "," << 0.10 << "," << densities[i] << "," << flows[i] << "," << speeds[i] << "\n";
        }
    }

    std::vector<double> ratios = {0.0, 0.05, 0.10, 0.15, 0.20};
    params.tauCar = 4;
    params.tauMotor = 1;
    for (double r : ratios) {
        run_fd(densities, r, params, 2000, 200, rng, flows, speeds);
        for (size_t i = 0; i < densities.size(); ++i) {
            out << "ratio," << params.tauCar << "," << r << "," << densities[i] << "," << flows[i] << "," << speeds[i] << "\n";
        }
    }

    return 0;
}
"""

# Guardar el C++ en archivo
with open("ca_mixed.cpp", "w", encoding="utf-8") as f:
    f.write(cpp_code)

# Compilar y ejecutar en Colab
!g++ -O2 -std=c++17 ca_mixed.cpp -o ca_mixed
!./ca_mixed results.csv

RuntimeError: No se encontro g++. Instala MinGW-w64 o MSYS2 y reinicia VS Code. Luego vuelve a ejecutar esta celda.

## Interpretacion de graficos
Los diagramas que se generan son de **flujo vs densidad**. La densidad esta en veh/km/carril y el flujo en veh/h/carril.

**1) Experimento de duracion $\tau_c$ (autos)**
- En baja densidad, las curvas casi coinciden: hay espacio suficiente y el cambio de carril no limita el flujo.
- Cerca de la densidad critica, aumentar $\tau_c$ reduce el flujo maximo, porque la doble ocupacion bloquea carriles por mas tiempo.
- En congestion, mayor $\tau_c$ mantiene flujos mas bajos por mayor interferencia lateral.

**2) Experimento de penetracion de motos**
- En baja densidad, mas motos tiende a bajar el flujo maximo (menor $v_{max}$).
- En alta densidad, mas motos puede elevar el flujo, porque cambian de carril mas rapido y ocupan menos longitud.

Si quieres, tambien puedo agregar el grafico de **velocidad vs densidad** para comparar con las figuras del articulo.

In [ ]:
# Graficas desde el CSV generado por el C++
import csv
import numpy as np
import matplotlib.pyplot as plt

rows = []
with open("results.csv", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

def plot_experiment(exp_name, key_field, metric, ylabel, title_suffix):
    groups = {}
    for r in rows:
        if r["experiment"] != exp_name:
            continue
        key = r[key_field]
        groups.setdefault(key, {"density": [], "metric": []})
        groups[key]["density"].append(float(r["density"]))
        groups[key]["metric"].append(float(r[metric]))
    plt.figure(figsize=(7, 4))
    for key, data in groups.items():
        order = np.argsort(data["density"])
        d = np.array(data["density"])[order]
        m = np.array(data["metric"])[order]
        plt.plot(d, m, marker="o", label=f"{key_field}={key}")
    plt.xlabel("Densidad (veh/km/lane)")
    plt.ylabel(ylabel)
    plt.title(f"{title_suffix} ({exp_name})")
    plt.legend()
    plt.grid(True)
    plt.show()

# Flujo vs densidad
plot_experiment("tau", "tau_car", "flow", "Flujo (veh/h/lane)", "Flujo vs densidad")
plot_experiment("ratio", "motor_ratio", "flow", "Flujo (veh/h/lane)", "Flujo vs densidad")

# Velocidad vs densidad
plot_experiment("tau", "tau_car", "speed", "Velocidad media (km/h)", "Velocidad vs densidad")
plot_experiment("ratio", "motor_ratio", "speed", "Velocidad media (km/h)", "Velocidad vs densidad")

## Comparacion con los resultados del articulo
- **Duracion de cambio $\tau_c$:** el articulo reporta que al aumentar $\tau_c$ disminuyen la capacidad (flujo maximo) y la velocidad media, con efecto mas marcado en densidades medias. Las curvas generadas reproducen esa tendencia: las series con $\tau_c$ mayor presentan menor pico de flujo y menor velocidad a igual densidad.
- **Penetracion de motos:** el articulo encuentra que en baja densidad un mayor porcentaje de motos reduce el flujo (menor $v_{max}$), mientras que en alta densidad puede mejorar el flujo por cambios de carril mas rapidos. En las graficas, la pendiente y el nivel de las curvas siguen esa transicion cualitativa: penalizacion a baja densidad y mejora relativa en alta densidad.

Si necesitas una comparacion cuantitativa (valores puntuales o errores relativos), puedo agregar tablas a partir del CSV.

## Comparacion cuantitativa (tabla)
La tabla resume, para cada serie, el **flujo maximo** y la **densidad critica** (densidad en la que se alcanza ese maximo).
Tambien se reporta la **velocidad media** en esa densidad critica.

In [ ]:
from IPython.display import Markdown, display

def summarize_peaks(exp_name, key_field):
    series = {}
    for r in rows:
        if r["experiment"] != exp_name:
            continue
        key = r[key_field]
        series.setdefault(key, []).append(r)

    summary = []
    for key, items in series.items():
        items_sorted = sorted(items, key=lambda x: float(x["density"]))
        flows = [float(x["flow"]) for x in items_sorted]
        speeds = [float(x["speed"]) for x in items_sorted]
        densities = [float(x["density"]) for x in items_sorted]
        idx = int(np.argmax(flows))
        summary.append({
            key_field: key,
            "flow_max": flows[idx],
            "density_crit": densities[idx],
            "speed_at_crit": speeds[idx],
        })
    return summary

def markdown_table(title, rows, key_field):
    headers = [key_field, "flow_max", "density_crit", "speed_at_crit"]
    lines = [f"### {title}", "", "| " + " | ".join(headers) + " |", "|" + "|".join(["---"] * len(headers)) + "|"]
    for r in sorted(rows, key=lambda x: float(x[key_field])):
        lines.append(
            f"| {r[key_field]} | {r['flow_max']:.2f} | {r['density_crit']:.2f} | {r['speed_at_crit']:.2f} |"
        )
    return "\n".join(lines)

tau_summary = summarize_peaks("tau", "tau_car")
ratio_summary = summarize_peaks("ratio", "motor_ratio")

display(Markdown(markdown_table("Resumen por duracion de cambio (tau_car)", tau_summary, "tau_car")))
display(Markdown(markdown_table("Resumen por penetracion de motos (motor_ratio)", ratio_summary, "motor_ratio")))